# Bits e bytes em C

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Ler, ligar, desligar e inverter bits com &, |, ^, ~ e deslocamentos, em tipos sem sinal de largura fixa como uint32_t.
- Empacotar vários campos pequenos num inteiro e desempacotá-los de novo com máscaras e deslocamentos.
- Olhar os bytes e bits de qualquer valor, como um float, copiando-o para um inteiro com memcpy.

**Neste notebook**, uma ideia de cada vez (rode cada célula antes de seguir):
1. Vendo os bits
2. &, |, ^ e ~: bit a bit
3. Deslocamentos: << e >>
4. Ligar, desligar, inverter e testar um bit
5. Máscaras: lendo um campo de bits
6. Tipos de tamanho fixo
7. Os bytes de um valor
8. Erros comuns

Os **exercícios** deste notebook terminam com linhas `check(...)`: testes mínimos que imprimem ✓ quando o seu código está certo. Rode a próxima célula uma vez antes, para eles poderem usar o `check`.

In [ ]:
// shared: check_int e check_str imprimem ✓ ou ✗; os exercícios abaixo usam
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado %ld, veio %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: esperado \"%s\", veio \"%s\"\n", what, expected, actual);
    }
}

## 1. Vendo os bits

Todo inteiro é guardado como bits. O C deixa você escrever números em hexadecimal (`0x2A`), em que cada dígito hexa é exatamente 4 bits, e imprimi-los com `%X`. `%08X` completa com zeros até 8 dígitos. Não existe código do `printf` para binário, então a célula abaixo define um pequeno `print_bits`.

In [ ]:
#include <stdint.h>
#include <stdio.h>

void print_bits(uint8_t v) {
    for (int i = 7; i >= 0; i--) {
        putchar((v >> i) & 1 ? '1' : '0');
    }
}

int main(void) {
    uint8_t values[] = {5, 0x2A, 255, 128};
    for (size_t i = 0; i < sizeof values / sizeof values[0]; i++) {
        printf("%3d = 0x%02X = ", values[i], values[i]);
        print_bits(values[i]);
        printf("\n");
    }
    return 0;
}

🧪 **Experimente:** acrescente `0x0F` e `0xF0` ao array: cada dígito hexa é um grupo de 4 bits.

## 2. &, |, ^ e ~: bit a bit

Esses operadores trabalham em cada par de bits separadamente:

| Operador | O bit é 1 quando… | Uso típico |
|---|---|---|
| `a & b` (e) | os dois bits são 1 | ficar só com alguns bits (uma máscara) |
| `a \| b` (ou) | pelo menos um é 1 | ligar bits |
| `a ^ b` (ou exclusivo) | os bits são diferentes | inverter bits |
| `~a` (não) | o bit era 0 | inverter tudo |

Não confunda com `&&` e `||`, que trabalham com valores inteiros de verdadeiro/falso.

In [ ]:
#include <stdint.h>
#include <stdio.h>

void print_bits(uint8_t v) {
    for (int i = 7; i >= 0; i--) {
        putchar((v >> i) & 1 ? '1' : '0');
    }
}

void show(const char *label, uint8_t v) {
    printf("%-6s ", label);
    print_bits(v);
    printf("\n");
}

int main(void) {
    uint8_t a = 0xCA;   // 11001010
    uint8_t b = 0x0F;   // 00001111
    show("a", a);
    show("b", b);
    show("a & b", a & b);
    show("a | b", a | b);
    show("a ^ b", a ^ b);
    show("~a", (uint8_t) ~a);
    return 0;
}

🧪 **Experimente:** preveja cada linha bit a bit antes de rodar. Depois calcule `a ^ b ^ b`: o que volta?

## 3. Deslocamentos: << e >>

`x << n` move cada bit n posições para a esquerda, preenchendo com zeros: multiplica por 2ⁿ. `x >> n` os move para a direita: para valores unsigned, divide por 2ⁿ, descartando o resto. `1u << k` é o número só com o bit k ligado, a peça básica de todo truque com bits.

In [ ]:
unsigned x = 13;
printf("%u << 1 = %u, %u << 3 = %u\n", x, x << 1, x, x << 3);
printf("%u >> 1 = %u, %u >> 2 = %u\n", x, x >> 1, x, x >> 2);
for (int k = 0; k < 5; k++) {
    printf("1u << %d = %u\n", k, 1u << k);
}

🧪 **Experimente:** confira que `x >> 2` é igual a `x / 4` para alguns valores de x.

## 4. Ligar, desligar, inverter e testar um bit

Com um 1 deslocado como máscara, quatro linhas resolvem um bit k:

| Objetivo | Código |
|---|---|
| ligar | `flags \|= 1u << k;` |
| desligar | `flags &= ~(1u << k);` |
| inverter | `flags ^= 1u << k;` |
| testar | `(flags >> k) & 1` |

É assim que programas guardam várias opções de sim/não (permissões, configurações) num inteiro só.

In [ ]:
uint32_t flags = 0;
flags |= 1u << 3;          // liga
flags |= 1u << 0;          // liga
printf("depois de ligar os bits 3 e 0: %u\n", (unsigned) flags);
flags ^= 1u << 0;          // inverte
flags &= ~(1u << 5);       // desliga (já estava desligado)
printf("agora: %u, bit 3 é %u, bit 0 é %u\n", (unsigned) flags, (unsigned) ((flags >> 3) & 1), (unsigned) ((flags >> 0) & 1));

🧪 **Experimente:** ligue também os bits 1 e 2, e preveja o número antes de rodar (2 + 4 + 8).

### ✏️ Exercício: Ligue um bit

Devolva x com o bit k em 1, usando `|` e um 1 deslocado. Um bit que já está ligado continua ligado.

In [ ]:
unsigned set_bit(unsigned x, int k) {
    // TODO: faça OR de x com 1u << k
    return x;
}

int main(void) {
    check_int("bit 3 de 0", set_bit(0, 3), 8);
    check_int("já ligado", set_bit(8, 3), 8);
    check_int("bit 4 de 1", set_bit(1, 4), 17);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
unsigned set_bit(unsigned x, int k) {
    return x | (1u << k);
}

int main(void) {
    check_int("bit 3 de 0", set_bit(0, 3), 8);
    check_int("já ligado", set_bit(8, 3), 8);
    check_int("bit 4 de 1", set_bit(1, 4), 17);
    return 0;
}
```

</details>

## 5. Máscaras: lendo um campo de bits

Hardware e formatos de arquivo costumam empacotar vários números pequenos num inteiro só. Para ler um campo de w bits que começa no bit s: **desloque s para a direita** para trazê-lo para baixo, depois **faça AND com uma máscara de w uns** (`(1u << w) - 1`, como `0x3F` para 6 bits). Os slides fazem isso com `0xABCD`.

#### 🎞️ Lendo um campo de bits com shift e máscara

![Para ler os bits 5 a 10 de 0xABCD, desloque 5 para a direita para trazê-los para baixo e faça AND com 0x3F para ficar com 6 bits: 011110, ou 30](shift-mask-steps.pt-br.svg)

In [ ]:
// Uma data em 16 bits, como os discos antigos guardavam: 7 bits para o ano (desde 1980), 4 para o mês, 5 para o dia.
uint16_t date = 0x5D7B;
unsigned day = date & 0x1F;
unsigned month = (date >> 5) & 0x0F;
unsigned year = 1980 + (date >> 9);
printf("%02u/%02u/%u\n", day, month, year);

🧪 **Experimente:** escreva à mão os 16 bits de outra data, como 1º de janeiro de 2000 (campo do ano 20, mês 1, dia 1), e confira que a célula a lê de volta.

### ✏️ Exercício: O nibble de baixo

Devolva os 4 bits mais baixos de x (um dígito hexa) com uma máscara.

In [ ]:
unsigned low_nibble(unsigned x) {
    // TODO: faça AND de x com 0xF
    return 0;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}

<details>
<summary>Ver uma resposta</summary>

```c
unsigned low_nibble(unsigned x) {
    return x & 0xFu;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}
```

</details>

## 6. Tipos de tamanho fixo

Um `int` pode ter 2, 4 ou 8 bytes, dependendo da máquina. Para trabalhar com bits, use os tipos de `<stdint.h>`, cujo tamanho é garantido: `uint8_t`, `uint16_t`, `uint32_t`, `uint64_t` (e `int8_t`… para os com sinal). Os tipos unsigned dão a volta ao passar do máximo, e os deslocamentos deles são totalmente definidos.

In [ ]:
printf("uint8_t: %zu, uint16_t: %zu, uint32_t: %zu, uint64_t: %zu bytes\n",
       sizeof(uint8_t), sizeof(uint16_t), sizeof(uint32_t), sizeof(uint64_t));
uint8_t small = 250;
small += 10;
printf("250 + 10 num uint8_t: %u\n", (unsigned) small);
uint64_t big = 1;
big <<= 40;
printf("2^40 = %llu\n", (unsigned long long) big);

🧪 **Experimente:** tente `big <<= 70;` no lugar: deslocar pelo menos a largura é comportamento indefinido, e a verificação acusa.

## 7. Os bytes de um valor

O `memcpy` copia os bytes crus de um valor para um array de `unsigned char`, para você olhá-los um a um. Na maioria dos computadores (x86 e ARM), números de vários bytes são guardados em **little-endian**: o byte menos significativo vem primeiro. O mesmo truque revela os bits de um `float`.

In [ ]:
uint32_t value = 0x12345678;
unsigned char bytes[4];
memcpy(bytes, &value, sizeof value);
for (int i = 0; i < 4; i++) {
    printf("byte %d: 0x%02X\n", i, bytes[i]);
}
float f = 1.0f;
uint32_t bits;
memcpy(&bits, &f, sizeof bits);
printf("1.0f é guardado como 0x%08X\n", (unsigned) bits);

🧪 **Experimente:** veja os bits de `-1.0f` e `2.0f`: qual bit é o sinal?

## 8. Erros comuns

- **`&` no lugar de `&&`** (ou `|` no lugar de `||`): `1 & 2` é 0, enquanto `1 && 2` é verdadeiro.
- **Precedência:** `x & 1 == 0` quer dizer `x & (1 == 0)`. Use sempre parênteses: `(x & 1) == 0`.
- **Deslocar valores com sinal:** `-1 >> 1` ou `1 << 31` num `int` dependem do compilador ou são indefinidos. Use tipos unsigned.
- **Deslocar pela largura ou mais:** `1u << 32` num valor de 32 bits é indefinido.

In [ ]:
unsigned x = 6;
printf("x & 1 é %u, x && 1 é %d\n", x & 1, x && 1);
if ((x & 1) == 0) {
    printf("%u é par\n", x);
}

🧪 **Experimente:** tire os parênteses em volta de `x & 1` e leia o aviso, depois a saída.

## Em resumo

O C trabalha direto nos bits dos inteiros: `&` (e), `|` (ou), `^` (ou exclusivo), `~` (não), `<<` e `>>` (deslocamentos). Use tipos **sem sinal** de largura fixa de `<stdint.h>`, como `uint32_t`: o tamanho deles é certo, e os deslocamentos são totalmente definidos.

```c
#include <stdio.h>
#include <stdint.h>
#include <string.h>

int main(void) {
    uint32_t flags = 0;
    flags |= 1u << 3;                     // liga o bit 3
    flags ^= 1u << 0;                     // inverte o bit 0
    int bit3 = (flags >> 3) & 1;          // lê o bit 3
    flags &= ~(1u << 0);                  // desliga o bit 0
    printf("%u %d\n", (unsigned) flags, bit3);   // 8 1

float f = 1.0f;
    uint32_t bits;
    memcpy(&bits, &f, sizeof bits);       // os próprios bits do float
    printf("0x%08X\n", (unsigned) bits); // 0x3F800000
    return 0;
}
```

Uma **máscara** seleciona um campo: `(value >> 5) & 0x3F` pega os 6 bits que começam no bit 5.

**Cuidado:** desloque só valores sem sinal, por menos que a largura deles: os outros deslocamentos são comportamento indefinido ou dependem do compilador.

## Leituras

- [Bitwise Operations](https://beej.us/guide/bgc/html/split/bitwise-operations.html) (Beej's Guide to C Programming, chapter 24, 4 min): AND, OR, XOR e NOT, depois os deslocamentos, e o aviso sobre deslocar valores com sinal. Curto: os operadores são os mesmos que você usou em Fundamentos da Computação, agora em C.
- [Fixed Width Integer Types](https://beej.us/guide/bgc/html/split/fixed-width-integer-types.html) (Beej's Guide to C Programming, chapter 37, 5 min): Opcional: uint8_t, uint32_t e parentes, os limites deles, e os especificadores de formato no estilo PRIu32. Os compiladores do Sphinx também imprimem uint32_t com %u, que os desafios usam.

## Pratique

O notebook cobriu o básico. Agora aplique nos desafios da unidade, na barra lateral do Sphinx, onde o Submit roda todos os testes, inclusive os ocultos:

- **Contar os bits** · Fácil
- **Cores RGB565** · Fácil
- **Anatomia de um float** · Médio
- **CRC-32** · Difícil